
# Ravel Jeux d'eau, M.30 — tempo curve & 6-level boundaries

流程：
1. 从 ATEPP match 文件计算按谱对齐的 tempo 曲线（八分音符网格，单位 BPM）。
2. 对所有演奏取均值 tempo。
3. 分六层层级分句（`str_vec = [3,2,2,2,2,2]`）。
4. 可视化：均值曲线 + 各层断点；另附若干演奏叠加参考。

依赖：`numpy`, `pandas`, `matplotlib`, `scipy`。若未安装 `music21`，会自动回退到基于 match 的网格长度估计。


In [ ]:

from pathlib import Path
import sys, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# assume notebook is placed under MERIX SUBMISSION/Velocity
NOTEBOOK_DIR = Path().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
PIECE_DIR = REPO_ROOT / "ATEPP-1.2/ATEPP-1.2/Maurice_Ravel/Jeux_d'eau,_M._30"
TOKENIZER_ROOT = REPO_ROOT / "MERIX SUBMISSION/MIREX_Model/MIREX_Tokenizer"
sys.path.insert(0, str(TOKENIZER_ROOT))

print('repo_root:', REPO_ROOT)
print('piece_dir:', PIECE_DIR)


In [ ]:

from dataclasses import asdict

def read_tpqn(fmt3x_path: Path, default: float = 24.0) -> float:
    if not fmt3x_path.exists():
        return default
    m = re.search(r"TPQN:\s*(\d+)", fmt3x_path.read_text())
    return float(m.group(1)) if m else default


def estimate_grid_from_matches(piece_dir: Path, fmt3x_path: Path, shift: int = 0) -> int:
    tpqn = read_tpqn(fmt3x_path)
    match_files = [p for p in piece_dir.glob('*_match.txt') if not p.stem.endswith(('pre_match','err_match'))]
    max_idx = 0
    for mf in match_files:
        for ln in mf.read_text().splitlines():
            if not ln.strip() or ln.startswith('//') or ln.startswith('Missing'):
                continue
            parts = re.split(r"\s+", ln.strip())
            if len(parts) < 12:
                continue
            score_time = float(parts[8])
            beat_idx = int(np.floor((score_time / tpqn) * 2.0)) - shift
            if beat_idx > max_idx:
                max_idx = beat_idx
    return int(max_idx + 1)


def compute_score_grid(piece_dir: Path):
    fmt3x = piece_dir / 'score_fmt3x.txt'
    score_path = piece_dir / 'musicxml_cleaned.musicxml'
    score_shift = 0
    score_total_beats = None
    try:
        from tokenizer.score_tokenizer import MusicXMLTokenizer
        tok = MusicXMLTokenizer(str(score_path))
        df_notes = pd.DataFrame([asdict(n) for n in tok.tokenize_notes()])
        pos = df_notes['position'].astype(float).to_numpy()
        dur = df_notes['duration'].astype(float).to_numpy()
        end_pos = (pos + dur) * 2.0
        score_total_beats = int(np.ceil(end_pos.max()))
    except Exception as e:
        print('[WARN] MusicXML parse failed, fallback to match grid:', e)

    if score_total_beats is None:
        fmt3x = piece_dir / 'score_fmt3x.txt'
        score_total_beats = estimate_grid_from_matches(piece_dir, fmt3x, shift=score_shift)
        print(f'[FALLBACK] grid from matches -> {score_total_beats} beats')

    return score_total_beats, score_shift


def bpm_curve_aligned_to_score(match_path: Path, fmt3x_path: Path, score_total_beats: int, score_shift: int,
                               bpm_range=(0, 5000), smooth_window: int = 3):
    tpqn = read_tpqn(fmt3x_path, default=24.0)
    rows = []
    for ln in match_path.read_text().splitlines():
        if not ln.strip() or ln.startswith('//') or ln.startswith('Missing'):
            continue
        parts = re.split(r"\s+", ln.strip())
        if len(parts) < 12:
            continue
        rows.append({
            'onset': float(parts[1]),
            'score_time': float(parts[8]),
            'score_note': parts[9],
            'err_idx': parts[10],
        })
    if not rows:
        return None
    df = pd.DataFrame(rows)
    df = df[(df['score_note'] != '*') & (df['err_idx'] == '0')]
    if df.empty:
        return None

    df['score_eighth'] = (df['score_time'] / tpqn) * 2.0
    df['beat_idx'] = np.floor(df['score_eighth']).astype(int) - int(score_shift)
    df = df[(df['beat_idx'] >= 0) & (df['beat_idx'] < score_total_beats)]
    if df.empty:
        return None

    beat_time = df.groupby('beat_idx')['onset'].median().sort_index()
    full_idx = pd.Index(range(score_total_beats), name='beat_idx')
    beat_time = beat_time.reindex(full_idx).interpolate('linear', limit_direction='both')

    dt = beat_time.diff()
    tempo = 30.0 / dt
    lo, hi = bpm_range
    tempo = tempo[(tempo > lo) & (tempo < hi)]
    tempo = tempo.rolling(window=smooth_window, center=True, min_periods=1).mean()
    tempo = tempo.clip(upper=600)
    tempo = tempo.reindex(range(score_total_beats)).interpolate('linear', limit_direction='both')
    return tempo.to_numpy()


def load_tempo_arrays(piece_dir: Path):
    fmt3x = piece_dir / 'score_fmt3x.txt'
    score_total_beats, score_shift = compute_score_grid(piece_dir)
    match_files = [p for p in piece_dir.glob('*_match.txt') if not p.stem.endswith(('pre_match','err_match'))]
    tempo_arrays = {}
    failed = []
    for mf in sorted(match_files):
        arr = bpm_curve_aligned_to_score(mf, fmt3x, score_total_beats, score_shift, bpm_range=(0,5000), smooth_window=3)
        if arr is None or len(arr)==0:
            failed.append(mf.name)
            continue
        if len(arr) != score_total_beats:
            if len(arr) > score_total_beats:
                arr = arr[:score_total_beats]
            else:
                arr = np.pad(arr, (0, score_total_beats - len(arr)), mode='edge')
        tempo_arrays[mf.name] = arr
    meta = {
        'score_total_beats': score_total_beats,
        'score_shift': score_shift,
        'n_match': len(match_files),
        'failed': failed,
    }
    return tempo_arrays, meta


In [ ]:

# boundaries & hierarchy

def group_analysis_new(tempo_curve, ws, plot=False, return_pro=False):
    tempo = np.asarray(tempo_curve, dtype=float)
    n = len(tempo)
    ws = int(ws)
    if ws <= 1 or n == 0:
        return (np.array([], dtype=int), tempo * 0) if return_pro else np.array([], dtype=int)
    pad_left = (ws - 1) // 2 if ws % 2 == 1 else ws // 2
    pad_right = ws - 1 - pad_left
    padded = np.pad(tempo, (pad_left, pad_right), mode='constant', constant_values=0.0)
    pro = np.zeros(n, dtype=float)
    for i in range(n):
        window = padded[i:i+ws]
        nonzero = window != 0
        pro[i] = np.sqrt(np.sum(window[nonzero] ** 2) / nonzero.sum()) if np.any(nonzero) else 0.0
    min_width = int(np.ceil(ws / 2))
    peaks, _ = find_peaks(-pro, distance=min_width)
    locs = []
    half = (ws - 1) // 2 if (ws % 2 == 1) else ws // 2
    for p in peaks:
        left = max(0, p - half)
        right = min(n - 1, p + half)
        segment = tempo[left:right+1]
        if segment.size == 0:
            continue
        locs.append(left + int(np.argmin(segment)))
    locs = np.array(sorted(set(locs)), dtype=int)
    return (locs, pro) if return_pro else locs


def boundaries_to_mask(n_beats, boundary_indices):
    mask = np.zeros(n_beats, dtype=int)
    boundary_indices = np.asarray(boundary_indices, dtype=int)
    boundary_indices = boundary_indices[(boundary_indices >= 0) & (boundary_indices < n_beats)]
    mask[boundary_indices] = 1
    return mask


def _pad_to_multiple(x, m):
    x = np.asarray(x, dtype=float).reshape(-1)
    pad = (-len(x)) % m
    if pad == 0:
        return x
    return np.concatenate([x, np.zeros(pad, dtype=float)])


def group_analysis_hierarchy(tempo_curve, str_vec, enforce_nested=True):
    tempo = np.asarray(tempo_curve, dtype=float).reshape(-1)
    n = len(tempo)
    str_vec = np.asarray(str_vec, dtype=int).reshape(-1)
    L = len(str_vec)
    if n == 0:
        return np.zeros((L, 0), dtype=bool), {l: np.array([], dtype=int) for l in range(1, L + 1)}
    avg = np.nanmean(tempo)
    if not np.isfinite(avg) or avg == 0:
        avg = 1.0
    energy = [None] * (L + 1)
    energy2 = [None] * (L + 1)
    valleys = [None] * (L + 1)
    s1 = int(str_vec[0])
    pad_raw = _pad_to_multiple(tempo, s1)
    pad_norm = _pad_to_multiple(tempo / avg, s1)
    energy[1] = pad_raw.reshape((s1, -1), order='F')
    energy2[1] = pad_norm.reshape((s1, -1), order='F')
    v1, _ = find_peaks(-tempo)
    valleys[1] = v1 + 1
    for i in range(2, L + 1):
        t_eng = energy[i - 1].copy()
        t_eng[t_eng == 0] = np.nan
        t_norm = t_eng / avg
        eng2 = np.sqrt(np.nanmean(t_norm ** 2, axis=0)) - np.nanstd(t_norm, axis=0)
        vi, _ = find_peaks(-eng2)
        valleys[i] = vi + 1
        si = int(str_vec[i - 1])
        eng2_pad = _pad_to_multiple(eng2, si)
        energy[i] = _pad_to_multiple(np.sqrt(np.nanmean(t_eng ** 2, axis=0)), si).reshape((si, -1), order='F')
        energy2[i] = eng2_pad.reshape((si, -1), order='F')
    results_raw = np.zeros((L, n), dtype=bool)
    for top in range(L, 0, -1):
        roots = valleys[top]
        if roots is None or len(roots) == 0:
            continue
        trace = np.zeros((top, len(roots)), dtype=int)
        trace[0, :] = roots
        for rj in range(len(roots)):
            for i in range(2, top + 1):
                lvl = top + 1 - i
                parent_col = trace[i - 2, rj]
                colvec = energy2[lvl][:, parent_col - 1]
                row = int(np.nanargmin(colvec)) + 1
                trace[i - 1, rj] = int(str_vec[lvl - 1]) * (parent_col - 1) + row
        beats_1b = trace[top - 1, :]
        beats_0b = beats_1b - 1
        beats_0b = beats_0b[(beats_0b >= 0) & (beats_0b < n)]
        results_raw[top - 1, beats_0b] = True
    level_sets = {l: np.where(results_raw[l - 1])[0] for l in range(1, L + 1)}
    if enforce_nested:
        cum = np.zeros(n, dtype=bool)
        nested = {}
        for l in range(L, 0, -1):
            cum |= results_raw[l - 1]
            nested[l] = np.where(cum)[0]
        level_sets = nested
    return results_raw, level_sets


In [ ]:

tempo_arrays, meta = load_tempo_arrays(PIECE_DIR)
print('match files:', meta['n_match'], 'failed:', len(meta['failed']))
if meta['failed']:
    print('failed examples:', meta['failed'][:5])
print('score_total_beats:', meta['score_total_beats'])

mean_tempo = np.mean(np.stack(list(tempo_arrays.values())), axis=0)
print('mean tempo length:', len(mean_tempo))


In [ ]:

# per-performance boundaries (optional probability)
ws = 12
boundary_masks = []
for nm, curve in tempo_arrays.items():
    locs, _ = group_analysis_new(curve, ws=ws, plot=False, return_pro=True)
    boundary_masks.append(boundaries_to_mask(len(curve), locs))

boundary_prob = np.stack(boundary_masks, axis=0).mean(axis=0)

# hierarchical (6 levels)
str_vec = [3, 2, 2, 2, 2, 2]
_, level_sets = group_analysis_hierarchy(mean_tempo, str_vec, enforce_nested=True)

# plot: mean tempo with six levels of boundaries
colors = ['C1','C2','C3','C4','C5','C6']
plt.figure(figsize=(14,5))
plt.plot(mean_tempo, color='black', linewidth=1.2, label='mean tempo')
for l, col in zip(range(1, 7), colors):
    locs = level_sets.get(l, [])
    if len(locs):
        plt.scatter(locs, mean_tempo[locs], s=18, color=col, label=f'level {l}', alpha=0.8)
plt.title("Jeux d'eau: mean tempo with 6-level breaks")
plt.xlabel('Beat index (eighth grid)')
plt.ylabel('BPM')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# plot: boundary probability + peaks for reference
from scipy.signal import find_peaks
peaks, _ = find_peaks(boundary_prob, height=0.3, distance=4, prominence=0.05)
plt.figure(figsize=(14,3))
plt.plot(boundary_prob, label='boundary prob')
plt.scatter(peaks, boundary_prob[peaks], color='crimson', s=22, label='peaks')
plt.xlabel('Beat index')
plt.ylabel('Prob')
plt.title('Aggregated boundary probability')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



## 模型推理（沿用 `analysis.ipynb` 参数）

> 说明：需要 `music21`, `torch`, `yaml` 等依赖。当前环境若缺 `music21` 会报错；留作可运行脚本供后续执行。


In [ ]:

from pathlib import Path
import numpy as np
import sys, re

# paths
NOTEBOOK_DIR = Path().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
PIECE_DIR = REPO_ROOT / "ATEPP-1.2/ATEPP-1.2/Maurice_Ravel/Jeux_d'eau,_M._30"
score_xml = PIECE_DIR / 'musicxml_cleaned.musicxml'
out_dir = NOTEBOOK_DIR / 'beat_data_jeux_deau_levels'
out_dir.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / 'MERIX SUBMISSION/MIREX_Model'))
from tokenizer_beat import extract_score_tokens, build_note_features

# 1) score -> note feats / beat ids
beat_unit = 0.5  # eighth-note grid to match tempo curves
print('tokenizing score...', score_xml)
tokens, meta = extract_score_tokens(score_xml, expand_repeats=False)
note_feats, beat_ids, num_beats = build_note_features(tokens, beat_unit=beat_unit)
print('notes:', note_feats.shape[0], 'num_beats:', num_beats)

# 2) per performance, per level -> npz (boundary_probs 先置零；模型只用 note_feats/beat_ids)
str_vec = [3, 2, 2, 2, 2, 2]
match_files = [p for p in PIECE_DIR.glob('*_match.txt') if not p.stem.endswith(('pre_match','err_match'))]
for mf in sorted(match_files):
    perf_id = mf.stem.replace('_match', '')
    for level_idx, ws in enumerate(str_vec, 1):
        boundary_probs = np.zeros(num_beats, dtype=np.float32)
        out_path = out_dir / f"JDE_{perf_id}_L{level_idx}.npz"
        np.savez(
            out_path,
            note_feats=note_feats.astype(np.float32),
            beat_ids=beat_ids.astype(np.int32),
            boundary_probs=boundary_probs,
            num_beats=int(num_beats),
            beat_unit=float(beat_unit),
            level=int(level_idx),
            level_ws=int(ws),
            performer_id=str(perf_id),
            mazurka_id="JDE",
        )
print('written npz:', len(list(out_dir.glob('*.npz'))), '->', out_dir)


In [ ]:

import numpy as np
import torch, yaml
from pathlib import Path
from scipy.signal import find_peaks
import sys

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
MIREX_DIR = REPO_ROOT / 'MERIX SUBMISSION/MIREX_Model'
sys.path.insert(0, str(MIREX_DIR))
from infer_beat import build_model, apply_position_mode, normalize_feats, build_beat_windows, slice_by_beats

# copy of analyze_boundary_dist (do_plots=False for batch run)
def analyze_boundary_dist(config_path, ckpt_path, input_dir, pattern, output_head="dist", pred_height=0.05, pred_min_dist=6, pred_prominence=0.03, include_empty_beats=False):
    cfg = yaml.safe_load(Path(config_path).read_text())
    cfg.setdefault('model', {})['performer_cond'] = False
    ckpt_path = Path(ckpt_path)
    files = sorted(Path(input_dir).glob(pattern))
    if not files:
        raise RuntimeError('No npz files found for pattern ' + pattern)
    sample = np.load(files[0])
    input_dim = sample['note_feats'].shape[1]
    model = build_model(cfg, input_dim=input_dim)
    state = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state, strict=False)
    model.eval()
    position_mode = cfg.get('data', {}).get('position_mode', 'absolute')
    value_ranges = cfg.get('data', {}).get('value_ranges')
    window_beats = cfg.get('data', {}).get('beat_sequence_length')
    window_stride = cfg.get('data', {}).get('beat_stride')
    max_len = cfg.get('data', {}).get('max_len')

    def infer_probs(npz_path):
        npz = np.load(npz_path)
        note_feats = npz['note_feats']
        beat_ids = npz['beat_ids']
        if beat_ids.size == 0:
            return np.array([])
        num_beats = int(npz['num_beats']) if 'num_beats' in npz else int(beat_ids.max()) + 1
        windows = build_beat_windows(num_beats, int(window_beats), int(window_stride))
        sum_probs = np.zeros(num_beats, dtype=np.float64)
        count_probs = np.zeros(num_beats, dtype=np.int64)
        for start, end in windows:
            feats, ids = slice_by_beats(note_feats, beat_ids, start, end, max_len)
            if feats is None or ids is None or feats.shape[0] == 0:
                continue
            feats = apply_position_mode(feats, position_mode)
            if value_ranges:
                feats = normalize_feats(feats, value_ranges)
            feats_t = torch.tensor(feats).unsqueeze(0)
            beat_ids_t = torch.tensor(ids).unsqueeze(0)
            attn_mask = beat_ids_t >= 0
            with torch.no_grad():
                logits, _ = model(
                    feats_t,
                    beat_ids=beat_ids_t,
                    num_beats=end - start,
                    attn_mask=attn_mask,
                    labels=None,
                    output_head=output_head,
                    performer_ids=torch.tensor([0]),
                )
                probs_win = torch.sigmoid(logits).squeeze(0).cpu().numpy()
            idx = np.arange(start, end)
            valid = np.zeros(end - start, dtype=bool)
            if ids.size > 0:
                valid[np.unique(ids[ids >= 0])] = True
            sum_probs[idx[valid]] += probs_win[valid]
            count_probs[idx[valid]] += 1
        probs = np.zeros(num_beats, dtype=np.float64)
        seen = count_probs > 0
        probs[seen] = sum_probs[seen] / count_probs[seen]
        return probs.astype(np.float32)

    pred_probs = infer_probs(files[0])
    pred_peaks, _ = find_peaks(pred_probs, distance=pred_min_dist, height=pred_height, prominence=pred_prominence)
    return pred_probs, pred_peaks

# --- run for 6 levels ---
ckpt_root = MIREX_DIR / 'check/beat_mazurka'
levels = {
    1: {
        "config": MIREX_DIR / "config_beat_mazurka_level1_ratio.yaml",
        "ckpt": ckpt_root / "level_1/beat_mazurka_L1_20260131_005320/best.pt",
        "output_head": "ratio",
        "pred_height": 0.02, \"pred_min_dist\": 2, "pred_prominence": 0.03,
    },
    2: {
        "config": MIREX_DIR / "config_beat_mazurka_level2_ratio.yaml",
        "ckpt": ckpt_root / "level_2/beat_mazurka_L2_20260130_234527/best.pt",
        "output_head": "ratio",
        "pred_height": 0.04, \"pred_min_dist\": 6, "pred_prominence": 0.03,
    },
    3: {
        "config": MIREX_DIR / "config_beat_mazurka_level3_ratio.yaml",
        "ckpt": ckpt_root / "level_3/beat_mazurka_L3_20260130_231129/best.pt",
        "output_head": "ratio",
        "pred_height": 0.04, \"pred_min_dist\": 6, "pred_prominence": 0.03,
    },
    4: {
        "config": MIREX_DIR / "config_beat_mazurka_level4.yaml",
        "ckpt": ckpt_root / "level_4/beat_mazurka_L4_20260130_190213/best.pt",
        "output_head": "dist",
        "pred_height": 0.03, "pred_min_dist": 3, "pred_prominence": 0.01,
    },
    5: {
        "config": MIREX_DIR / "config_beat_mazurka_level5.yaml",
        "ckpt": ckpt_root / "level_5/beat_mazurka_L5_20260130_190152/best.pt",
        "output_head": "dist",
        "pred_height": 0.03, "pred_min_dist": 3, "pred_prominence": 0.01,
    },
    6: {
        "config": MIREX_DIR / "config_beat_mazurka_level6.yaml",
        "ckpt": ckpt_root / "level_6/beat_mazurka_L6_20260130_141117/best.pt",
        "output_head": "dist",
        "pred_height": 0.03, "pred_min_dist": 3, "pred_prominence": 0.01,
    },
}

infer_dir = NOTEBOOK_DIR / 'beat_data_atepp_levels'
results_pred = {}
for lvl in range(1,7):
    p = levels[lvl]
    probs, peaks = analyze_boundary_dist(
        p['config'], p['ckpt'], infer_dir,
        pattern=f"*_L{lvl}.npz",
        output_head=p['output_head'],
        pred_height=p['pred_height'], pred_min_dist=p['pred_min_dist'], pred_prominence=p['pred_prominence']
    )
    results_pred[lvl] = {'probs': probs, 'peaks': peaks}
    print(f"Level {lvl}: {len(peaks)} peaks ->", peaks)



## 预测 vs. 聚合真值可视化（分层）
参考 `analysis.ipynb`：用前面算的预测 `results_pred` 与基于滑窗断句概率的聚合真值对比。
- 真值：对全部演奏用 `group_analysis_new(ws=12)` 聚合得到 `boundary_prob`。
- 预测：各 level 的 `results_pred[level]['probs']` 和 `peaks`。


In [ ]:

import numpy as np, matplotlib.pyplot as plt
from scipy.signal import find_peaks
from pathlib import Path

# 若前面变量缺失，可重新加载 tempo_arrays
if 'tempo_arrays' not in globals():
    # reuse helpers defined earlier (read_tpqn, compute_score_grid, bpm_curve_aligned_to_score, load_tempo_arrays, group_analysis_new)
    NOTEBOOK_DIR = Path().resolve()
    REPO_ROOT = NOTEBOOK_DIR.parents[1]
    PIECE_DIR = REPO_ROOT / "ATEPP-1.2/ATEPP-1.2/Maurice_Ravel/Jeux_d'eau,_M._30"
    tempo_arrays, meta = load_tempo_arrays(PIECE_DIR)

# 聚合真值 boundary_prob
ws_true = 12
boundary_masks = []
for nm, curve in tempo_arrays.items():
    locs, _ = group_analysis_new(curve, ws=ws_true, plot=False, return_pro=True)
    mask = boundaries_to_mask(len(curve), locs)
    boundary_masks.append(mask)
boundary_prob = np.stack(boundary_masks, axis=0).mean(axis=0)
true_peaks, _ = find_peaks(boundary_prob, height=0.1, distance=4, prominence=0.05)

if 'results_pred' not in globals():
    raise RuntimeError('results_pred 未定义，请先运行上面的模型推理单元')

levels = sorted(results_pred.keys())
for lvl in levels:
    probs = results_pred[lvl]['probs']
    peaks = results_pred[lvl]['peaks']
    plt.figure(figsize=(12,4))
    plt.plot(boundary_prob, label='boundary_prob (agg true)', color='gray', alpha=0.7)
    plt.scatter(true_peaks, boundary_prob[true_peaks], color='black', s=18, label='true peaks')
    plt.plot(probs, label=f'pred probs (L{lvl})', color='C1')
    if len(peaks):
        plt.scatter(peaks, probs[peaks], color='red', s=22, label='pred peaks')
    plt.title(f"Jeux d'eau — Level {lvl}: Pred vs Agg True")
    plt.xlabel('Beat index (eighth grid)')
    plt.ylabel('Probability')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 按 analysis.ipynb 风格的可视化


In [ ]:

import numpy as np, matplotlib.pyplot as plt
from scipy.signal import find_peaks

if 'boundary_masks' not in globals():
    boundary_masks = []
    for nm, curve in tempo_arrays.items():
        locs, _ = group_analysis_new(curve, ws=ws_true, plot=False, return_pro=True)
        boundary_masks.append(boundaries_to_mask(len(curve), locs))
boundary_prob = np.stack(boundary_masks, axis=0).mean(axis=0)
true_counts = np.stack(boundary_masks, axis=0).sum(axis=0)
true_peaks, _ = find_peaks(boundary_prob, height=0.1, distance=4, prominence=0.05)
true_avg_peaks = true_counts.sum() / max(len(boundary_masks),1)

for lvl in sorted(results_pred.keys()):
    probs = results_pred[lvl]['probs']
    pred_peaks = results_pred[lvl]['peaks']
    pred_peak_strengths = probs[pred_peaks] if pred_peaks.size>0 else np.array([])
    if pred_peak_strengths.size>0:
        rel = pred_peak_strengths / pred_peak_strengths.sum()
        pred_peak_strengths_scaled = rel * true_avg_peaks
    else:
        pred_peak_strengths_scaled = np.array([])

    plt.figure(figsize=(12,6))
    plt.bar(np.arange(len(true_counts)), true_counts/len(boundary_masks), width=1.0, alpha=0.6, label='True peak frequency')
    for i, pk in enumerate(pred_peaks):
        plt.axvline(pk, color='red', linewidth=1.0, alpha=0.8, label='Pred peaks' if i==0 else None)
    plt.title(f"Level {lvl}: True distribution vs Pred peaks")
    plt.xlabel('Beat index'); plt.ylabel('True peak freq'); plt.legend(); plt.tight_layout(); plt.show()

    plt.figure(figsize=(12,4))
    plt.plot(probs, label='Pred probs', linewidth=1.2)
    plt.scatter(pred_peaks, probs[pred_peaks], color='red', s=20, label='Pred peaks')
    plt.title(f"Level {lvl}: Pred curve + peaks")
    plt.xlabel('Beat index'); plt.ylabel('Pred prob'); plt.legend(); plt.tight_layout(); plt.show()

    # aligned comparison (true peaks as x)
    true_rel = true_counts[true_peaks] / max(true_counts.sum(),1)
    pred_rel = []
    for t in true_peaks:
        assigned = np.where(np.abs(pred_peaks - t) <= 6)[0]
        if assigned.size==0:
            pred_rel.append(0.0)
        else:
            pred_rel.append(pred_peak_strengths_scaled[assigned].sum())
    pred_rel = np.array(pred_rel)
    plt.figure(figsize=(12,5))
    plt.plot(true_peaks, true_rel, label='True peak rel', linewidth=1.5)
    plt.plot(true_peaks, pred_rel, label='Pred rel (aligned)', linewidth=1.5)
    plt.title(f"Level {lvl}: Aligned Pred vs True Relative Strength")
    plt.xlabel('True peak beat index'); plt.ylabel('Relative strength'); plt.legend(); plt.tight_layout(); plt.show()

    plt.figure(figsize=(12,5))
    plt.plot(probs, label='Pred curve', linewidth=1.2, color='C0')
    plt.plot(true_peaks, true_rel, label='True relative strength', linewidth=1.2, color='C1')
    plt.title(f"Level {lvl}: Pred curve vs True rel strength")
    plt.xlabel('Beat index'); plt.ylabel('Value'); plt.legend(); plt.tight_layout(); plt.show()



## 分层真值 vs 预测（各层独立概率对比）
- 对每个演奏先跑 `group_analysis_hierarchy(str_vec=[3,2,2,2,2,2])`，生成各层的真值断句 mask，再对演奏取平均得到 `true_prob_level[l]`。
- 与对应 level 的预测概率 `results_pred[l]['probs']` 对比，同图显示曲线和峰值。


In [ ]:

import numpy as np, matplotlib.pyplot as plt
from scipy.signal import find_peaks

str_vec = [3,2,2,2,2,2]
true_prob_level = {}

# 逐演奏分层断句 -> 聚合概率
for l in range(1, 7):
    masks = []
    for nm, curve in tempo_arrays.items():
        _, level_sets_nm = group_analysis_hierarchy(curve, str_vec, enforce_nested=True)
        locs = level_sets_nm.get(l, np.array([], dtype=int))
        masks.append(boundaries_to_mask(len(curve), locs))
    true_prob_level[l] = np.stack(masks, axis=0).mean(axis=0)

for l in sorted(results_pred.keys()):
    pred_probs = results_pred[l]['probs']
    true_probs = true_prob_level[l]
    true_peaks, _ = find_peaks(true_probs, height=0.05, distance=4, prominence=0.02)
    pred_peaks = results_pred[l]['peaks']
    plt.figure(figsize=(12,4))
    plt.plot(true_probs, label=f'True prob (L{l})', color='C0')
    plt.scatter(true_peaks, true_probs[true_peaks], color='C0', s=20, marker='x', label='True peaks')
    plt.plot(pred_probs, label=f'Pred prob (L{l})', color='C1')
    if len(pred_peaks):
        plt.scatter(pred_peaks, pred_probs[pred_peaks], color='C1', s=22, marker='o', facecolors='none', label='Pred peaks')
    plt.title(f"Level {l}: True vs Pred probability")
    plt.xlabel('Beat index (eighth grid)')
    plt.ylabel('Probability')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 重新生成 ATEPP 特征 (beat_unit=0.5)
- 依赖 music21；需已安装才能运行。
- 输出到 `MERIX SUBMISSION/Velocity/beat_data_atepp_levels/`，每场演奏每层一份 npz，字段与 Mazurka 数据格式一致（boundary_probs 先置 0）。


In [ ]:
from pathlib import Path
import numpy as np
import sys

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[1]
ATEPP_ROOT = REPO_ROOT / "ATEPP-1.2/ATEPP-1.2"
OUT_DIR = NOTEBOOK_DIR / "beat_data_atepp_levels"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# tokenizer_beat requires music21
sys.path.insert(0, str(REPO_ROOT / "MERIX SUBMISSION/MIREX_Model"))
# add tokenizer module path
sys.path.insert(0, str(REPO_ROOT / "MERIX SUBMISSION/MIREX_Model/MIREX_Tokenizer"))
from tokenizer_beat import extract_score_tokens, build_note_features

BEAT_UNIT = 0.5  # eighth-note grid
STR_VEC = [3,2,2,2,2,2]

xml_paths = sorted(ATEPP_ROOT.rglob("musicxml_cleaned.musicxml"))
print("found xml:", len(xml_paths))

count_npz = 0
for xml_path in xml_paths:
    piece_dir = xml_path.parent
    tokens, meta = extract_score_tokens(xml_path, expand_repeats=False)
    note_feats, beat_ids, num_beats = build_note_features(tokens, beat_unit=BEAT_UNIT)
    match_files = [p for p in piece_dir.glob("*_match.txt") if not p.stem.endswith(("pre_match","err_match"))]
    for mf in match_files:
        perf_id = mf.stem.replace("_match", "")
        base = piece_dir.name.replace(' ', '_') + "_" + perf_id
        for level_idx, ws in enumerate(STR_VEC, 1):
            out_path = OUT_DIR / f"{base}_L{level_idx}.npz"
            boundary_probs = np.zeros(num_beats, dtype=np.float32)
            np.savez(
                out_path,
                note_feats=note_feats.astype(np.float32),
                beat_ids=beat_ids.astype(np.int32),
                boundary_probs=boundary_probs,
                num_beats=int(num_beats),
                beat_unit=float(BEAT_UNIT),
                level=int(level_idx),
                level_ws=int(ws),
                performer_id=str(perf_id),
                mazurka_id=piece_dir.name,
            )
            count_npz += 1
print("written", count_npz, "npz to", OUT_DIR)
